In [ ]:
import sys
sys.path.append("..")      # lets you import from src/

import pandas as pd
import matplotlib.pyplot as plt
from src.data_loader import *

In [ ]:
for s in DATASETS:
    files = list_files(s)
    x = load_snapshot(files[0])
    print(s, len(files), x.shape, file_timestamp(files[0]), "->", file_timestamp(files[-1]))

In [ ]:
df = pd.read_parquet("../data/processed_features.parquet")

d = df[df["set"] == "set2"]
for b, g in d.groupby("bearing"):
    plt.plot(g["idx"], g["rms"], label=b)
plt.legend(); plt.xlabel("snapshot"); plt.ylabel("RMS"); plt.title("Set 2: RMS over life")
plt.show()

In [ ]:
healthy = g[g["idx"] < 0.25 * len(g)]
z = (g["rms"] - healthy["rms"].mean()) / healthy["rms"].std()

In [ ]:
g = df[(df["set"] == "set2") & (df["bearing"] == "b4")].tail(5)
print(g[["idx", "timestamp", "rms", "peak", "std"]])

x = load_snapshot(list_files("set2")[-1])
print(x.shape, x.mean(axis=0), x.std(axis=0))

In [ ]:
d = df[df["set"] == "set2"]
for b, g in d.groupby("bearing"):
    plt.plot(g["idx"], g["rms"], label=b)
plt.legend(); plt.xlabel("snapshot"); plt.ylabel("RMS"); plt.title("Set 2: RMS over life")

plt.savefig("../results/set2_rms.png", dpi=150, bbox_inches="tight")   # before show
plt.show()

Fault frequencies

The IMS rig runs at 2000 RPM with Rexnord ZA-2115 double-row bearings (16 rollers per row, pitch diameter 2.815", roller diameter 0.331", contact angle 15.17°). Compute the frequencies from the geometry instead of hard-coding them, so the report can show the derivation:

Fault	                    Formula	                  Value
BPFO (outer race)	(n/2)·f_r·(1 − d/D·cosθ)	    ≈ 236.4 Hz
BPFI (inner race)	(n/2)·f_r·(1 + d/D·cosθ)	    ≈ 296.9 Hz
BSF (roller)	    (D/2d)·f_r·(1 − (d/D·cosθ)²)	≈ 139.9 Hz
FTF (cage)	        ½·f_r·(1 − d/D·cosθ)	        ≈ 14.8 Hz

Your four failures map to these: Set 2 B1 and Set 3 B3 fail at the outer race (BPFO), Set 1 B3 at the inner race (BPFI), and Set 1 B4 at a roller (BSF).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from src.data_loader import *
from src.features import envelope_spectrum, FAULT_FREQS

files = list_files("set2")
ch = channel_map("set2")["b1"][0]     # B1 is the failed bearing

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for ax, i, label in zip(axes, [100, 975], ["early (healthy)", "late (near failure)"]):
    x = load_snapshot(files[i])[:, ch]
    f, E = envelope_spectrum(x)
    ax.plot(f, E, lw=0.8)
    ax.set_xlim(0, 1000)
    ax.set_title(f"Set 2 B1, snapshot {i}: {label}")
    for h in (1, 2, 3):
        ax.axvline(h * FAULT_FREQS["bpfo"], color="r", ls="--", alpha=0.6,
                   label="BPFO harmonics" if h == 1 else None)
axes[0].legend(); axes[1].set_xlabel("Frequency (Hz)")
plt.tight_layout()
plt.savefig("../results/set2_b1_envelope_spectrum.png", dpi=150)
plt.show()

In [ ]:
import numpy as np
from src.data_loader import *
from src.features import envelope_spectrum, FAULT_FREQS

def observed_peak(set_name, bearing, snap_idx, fault, ch_pos=0, half_width=0.06):
    files = list_files(set_name)
    ch = channel_map(set_name)[bearing][ch_pos]
    f, E = envelope_spectrum(load_snapshot(files[snap_idx])[:, ch])
    nominal = FAULT_FREQS[fault]
    m = (f > nominal * (1 - half_width)) & (f < nominal * (1 + half_width))
    f0 = f[m][E[m].argmax()]
    snr = E[m].max() / np.median(E[(f > 10) & (f < 1000)])
    print(f"{set_name} {bearing} {fault}: peak {f0:.1f} Hz, scale {f0/nominal:.4f}, SNR {snr:.1f}")

observed_peak("set3", "b3", -20, "bpfo")        # outer race, near end of life
observed_peak("set1", "b3", -50, "bpfi")        # inner race
observed_peak("set1", "b3", -50, "bpfi", ch_pos=1)
observed_peak("set1", "b4", -50, "bsf")         # roller (expect weak)
observed_peak("set1", "b4", -50, "bsf", ch_pos=1)

In [ ]:
df = pd.read_parquet("../data/processed_features.parquet")
spec = np.load("../data/env_spectra.npy")

row = df[(df["set"] == "set2") & (df["bearing"] == "b1") & (df["idx"] == 975)].index[0]
freqs = np.linspace(0, 1000, spec.shape[1])   # ≈0.98 Hz bins
plt.plot(freqs, spec[row])

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import importlib, src.features
importlib.reload(src.features)
from src.features import FAULT_FREQS, SPEED_SCALE

df = pd.read_parquet("../data/processed_features.parquet")
spec = np.load("../data/env_spectra.npy")
freqs = np.linspace(0, 1000, spec.shape[1])

# (a) SNR of every fault over time, smoothed
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for ax, (b, ch) in zip(axes.ravel(), [("b3", 4), ("b3", 5), ("b4", 6), ("b4", 7)]):
    g = df[(df["set"] == "set1") & (df["channel"] == ch)]
    for fault in ("bpfo", "bpfi", "bsf"):
        ax.semilogy(g["idx"], g[f"env_{fault}_snr"].rolling(15, center=True).median(),
                    label=fault, lw=1)
    ax.set_title(f"Set 1 {b}, channel {ch}")
axes[0, 0].legend()
plt.tight_layout(); plt.savefig("../results/set1_fault_snr_over_time.png", dpi=150)

# (b) averaged spectra: healthy vs. late life (averaging 100 snapshots reduces noise)
n = df[df["set"] == "set1"]["idx"].max() + 1

def avg_spec(ch, lo, hi):
    rows = df[(df["set"] == "set1") & (df["channel"] == ch)
              & (df["idx"] >= lo) & (df["idx"] < hi)].index
    return spec[rows].mean(axis=0)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for ax, ch, title in zip(axes, (4, 6), ("B3 (inner race)", "B4 (roller)")):
    ax.plot(freqs, avg_spec(ch, 100, 200), label="early", lw=0.8)
    ax.plot(freqs, avg_spec(ch, n - 102, n - 2), label="last 100 (excl. final 2)", lw=0.8)
    for name, c in (("bpfo", "r"), ("bpfi", "g"), ("bsf", "m")):
        for h in (1, 2, 3):
            ax.axvline(h * FAULT_FREQS[name] * SPEED_SCALE, color=c, ls="--", alpha=0.5,
                       label=name if h == 1 else None)
    ax.set_title(f"Set 1 {title}"); ax.set_xlim(0, 1000)
axes[0].legend(ncol=5)
plt.tight_layout(); plt.savefig("../results/set1_avg_envelope.png", dpi=150)

In [ ]:
df["env_fault_max_snr"] = df[["env_bpfo_snr", "env_bpfi_snr", "env_bsf_snr"]].max(axis=1)

In [ ]:
import numpy as np, pandas as pd
from src.features import add_spectral_features

df = pd.read_parquet("../data/processed_features.parquet")
spec = np.load("../data/env_spectra.npy")

df = add_spectral_features(df, spec)

In [ ]:
df.to_parquet("../data/processed_features.parquet", index=False)

In [ ]:
from scipy.signal import find_peaks
from src.features import SPEED_SCALE, FAULT_FREQS

df = pd.read_parquet("../data/processed_features.parquet")
spec = np.load("../data/env_spectra.npy")
freqs = np.linspace(0, 1000, spec.shape[1])
df1 = df[df["set"] == "set1"]
n = df1["idx"].max() + 1
fr = 2000 / 60 * SPEED_SCALE

def avg_spec(ch, lo, hi):
    rows = df1[(df1["channel"] == ch) & (df1["idx"] >= lo) & (df1["idx"] < hi)].index
    return spec[rows].mean(axis=0)

def refine(s, i):                       # parabolic interpolation -> sub-bin frequency
    a, b, c = s[i - 1], s[i], s[i + 1]
    return (i + 0.5 * (a - c) / (a - 2 * b + c)) * (freqs[1] - freqs[0])

# 1) What are B3's peaks, in units of shaft frequency?
s = avg_spec(4, n - 102, n - 2)
pk, _ = find_peaks(s, height=np.median(s) * 2.5, distance=5)
for p in pk[freqs[pk] < 700]:
    f0 = refine(s, p)
    print(f"{f0:7.1f} Hz  amp {s[p]:.4f}  = {f0 / fr:5.2f} x shaft")

print("\nBPFI scaled:", round(FAULT_FREQS['bpfi'] * SPEED_SCALE, 1), "Hz =",
      round(FAULT_FREQS['bpfi'] * SPEED_SCALE / fr, 2), "x shaft")

# 2) B4: cage frequency or half shaft speed?
s4 = avg_spec(6, n - 102, n - 2)
m = (freqs > 8) & (freqs < 25)
i = np.where(m)[0][s4[m].argmax()]
print(f"\nB4 low peak: {refine(s4, i):.1f} Hz | FTF {FAULT_FREQS['ftf']*SPEED_SCALE:.1f} Hz | 0.5x shaft {fr/2:.1f} Hz")

In [ ]:
def shaft_comb_snr(spec, freqs, fr, n_harm=30, tol=1.5):
    noise = np.median(spec[:, (freqs > 10) & (freqs < 1000)], axis=1) + 1e-12
    amps = []
    for k in range(1, n_harm + 1):
        m = np.abs(freqs - k * fr) <= tol
        amps.append(spec[:, m].max(axis=1))
    return np.sum(amps, axis=0) / (n_harm * noise)

df["env_shaft_comb_snr"] = shaft_comb_snr(spec, freqs, fr)

In [ ]:
freqs = np.arange(spec.shape[1]) * 20000 / 20480      # exact bin frequencies

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, ch in zip(axes, (4, 5)):                      # both accelerometers on B3
    ax.plot(freqs, avg_spec(ch, n - 102, n - 2), lw=1)
    ax.axvline(FAULT_FREQS["bpfi"] * 0.9804, color="g", ls="--", label="BPFI (~291 Hz)")
    ax.axvline(9 * 32.68, color="k", ls=":", label="9 x shaft (294 Hz)")
    ax.set_xlim(275, 310); ax.set_title(f"B3 channel {ch}")
axes[0].legend()
plt.show()

In [ ]:
df = pd.read_parquet("../data/processed_features.parquet")
spec = np.load("../data/env_spectra.npy")
df = add_spectral_features(df, spec)

feats = ["rms", "crest", "env_bpfi_snr", "env_ftf_snr", "env_shaft_comb_snr"]
# crest instead of kurt: scipy's kurtosis is "excess" (about 0 when healthy), so ratios to baseline blow up

fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
for ax, (ch, name) in zip(axes.ravel(), [(4, "B3 ch4"), (5, "B3 ch5"), (6, "B4 ch6"), (7, "B4 ch7")]):
    g = df[(df["set"] == "set1") & (df["channel"] == ch)].sort_values("idx")
    base = g[g["idx"] < 0.25 * len(g)]
    for f in feats:
        y = g[f] / base[f].median()
        ax.semilogy(g["idx"], y.rolling(15, center=True).median(), label=f, lw=1)
    ax.set_title(f"Set 1 {name}: feature / healthy-baseline median")
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.savefig("../results/set1_feature_comparison.png", dpi=150)

In [ ]:
import matplotlib.pyplot as plt
from src.health import prepare

plot_features = ["rms", "peak", "kurt", "env_bsf"]
runs = prepare(df, plot_features)
r = runs["set3_b3"]
fig, ax = plt.subplots(figsize=(10, 4))
for f in ["rms", "peak", "kurt", "env_bsf"]:
    ax.plot(r["life"], r[f], label=f, lw=0.8)
ax.axvspan(0.15, 0.40, color="grey", alpha=0.15, label="baseline window")
ax.axhline(5, color="k", ls=":")
ax.set_yscale("symlog", linthresh=5)
ax.set_xlabel("fraction of life"); ax.set_ylabel("z-score"); ax.legend()
plt.show()

In [ ]:
r = df[(df["set"] == "set3") & (df["bearing"] == "b3")].groupby("idx")["rms"].mean()
life = np.arange(len(r)) / (len(r) - 1)
print("baseline (0.15-0.40):", r[(life >= .15) & (life < .40)].median())
print("plateau  (0.55-0.90):", r[(life >= .55) & (life < .90)].median())
print("end      (last 2%):  ", r[life > .98].median())

In [ ]:
import numpy as np, pandas as pd
from src.features import add_spectral_features
from src.health import prepare, summarize

df = pd.read_parquet("../data/processed_features.parquet")
spec = np.load("../data/env_spectra.npy")
df = add_spectral_features(df, spec)

meta = ["set", "bearing", "channel", "idx", "timestamp", "failed"]
features = [c for c in df.columns if c not in meta]

runs = prepare(df, features)
summary = summarize(runs, features)
summary.sort_values("mono", ascending=False)

## Phase 3: anomaly onset with an effect-size rule

In [ ]:
from src.labels import run_table, find_onset, RUNS

tables = {f"{s}_{b}": run_table(df, s, b) for s, b in RUNS}

rows = []
for ratio_thr in (1.1, 1.15, 1.25, 1.5, 2.0):
    for n in (10, 20, 50):
        row = {"ratio_thr": ratio_thr, "n_consec": n}
        for name, u in tables.items():
            i = find_onset(u, ratio_thr, n)
            row[name] = np.nan if i is None else round(u.loc[i, "life"], 2)
        rows.append(row)
pd.DataFrame(rows)

In [ ]:
for name, u in tables.items():
    i = find_onset(u, 1.25, 20)
    print(name, "onset life:", None if i is None else round(u.loc[i, "life"], 2),
          "| hours onset->end:", None if i is None else round(u.loc[i, "rul_hours"], 1),
          "| total hours:", round(u["rul_hours"].iloc[0], 1))